In [1]:
pip install --upgrade gradio

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/31.0 MB ? eta -:--:--
    --------------------------------------- 0.5/31.0 MB 3.1 MB/s eta 0:00:10
   - -------------------------------------- 1.0/31.0 MB 3.2 MB/s eta 0:00:10
   -- ------------------------------------- 1.8/31.0 MB 3.4 MB/s eta 0:00:09
   --- ------------------------------------ 2.6/31.0 MB 3.3 MB/s eta 0:00:09
   ---- ----------------------------------- 3.4/31.0 MB 3.4 MB/s eta 0:00:09
   ----- ---------------------------------- 4.2/31.0 MB 3.4 MB/s eta 0:00:08
   ------ --------------------------------- 4.7/31.0 MB 3.5 MB/s eta 0:00:08
   ------- -------------------------------- 5.8/31.0 MB 3.5 MB/s eta 0:00:08
   -------- ------------------------------- 6.6/3


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [59]:
from openai import OpenAI
import gradio as gr
from ollama import Client
from constent import fetch_website_contents
print(gr.__version__)


6.20.0


In [4]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL_OLLAMA = 'llama3.2:1b'
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL,api_key='ollama')

In [5]:
system_message = 'you are a helpful assistant'

def message_ollama(prompt):
    messages=[
        {'role':'system','content':system_message},
        {'role':'user','content':prompt}
        ]
    response = ollama.chat.completions.create(model=MODEL_OLLAMA,messages=messages)
    return response.choices[0].message.content


In [7]:
message_ollama("What is today's date?")

'However, I\'m a large language model, I don\'t have real-time access to your location or current date. But I can suggest some ways for you to find out the current date.\n\nYou can:\n\n1. Check your smartphone or computer\'s clock or calendar app.\n2. Look at a physical schedule or planner.\n3. Search online for "current date" or "today\'s date".\n4. Ask a voice assistant like Siri, Google Assistant, or Alexa.\n\nPlease let me know if you need help with anything else!'

## User Interface Using Gradio Framwork

In [12]:
# create the interface
# using share true means that it can be accessed publically (view_ui.launch(share=True))
#when we run this it creats this url  Running on public URL: https://2508d1a30bc8665ebd.gradio.live
# when we use view_ui.launch(inbrowser=True) it get open in browser on local host like this Running on local URL:  http://127.0.0.1:7862


view_ui = gr.Interface(
    fn=message_ollama,
    inputs="textbox",
    outputs='textbox',
    flagging_mode='never'
)
view_ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Adding authentication 
when we mention email passowrd in launch it display login page 

In [13]:
view_login_page = gr.Interface(
    fn = message_ollama,
    inputs="textbox",
    outputs="textbox",
    flagging_mode='never'
)

view_login_page.launch(auth=('Faizan',"faizan@123"))

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Forcing dark mode
Based on system setting it is showing theme Light or Dark Mode 
we can change by wrighting some code 

In [26]:
view_login_page = gr.Interface(
    fn = message_ollama,
    inputs="textbox",
    outputs="textbox",
    flagging_mode='never',
    theme=gr.themes.Default()
    # theme=gr.themes.Soft()
    # theme=gr.themes.Monochrome()
    # theme=gr.themes.Glass()
)

view_login_page.launch()

c:\Users\LOG IN\Desktop\llm_engineering\.venv\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [32]:
# Make UI more intresting 
input_details=gr.Textbox(label='Your message',info='Enter a Message',lines=3)
output_details=gr.Textbox(label='Response',lines=7)

view_ui=gr.Interface(
    fn=message_ollama,
    title= 'Practise',
    inputs=[input_details],
    outputs=[output_details],
    examples=['Hello', 'Hi there'],
    flagging_mode='never'
)

view_ui.launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


In [33]:
# lets use Markdown Now as output details 
# this system message is an variable, and written above and here we are updating the same variable 

system_message = "You are a helpful assistant that responds in markdown without code blocks"

input_details=gr.Textbox(label='Your Message', info="Please Enter you message here", lines=2)
output_details=gr.Markdown(label='Response')

view_ui=gr.Interface(
    fn=message_ollama,
    inputs=[input_details],
    outputs=[output_details],
     examples=[
        "Explain the Transformer architecture in simple words",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ],
        flagging_mode='never'
)
view_ui.launch()


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# create a function and get response in streams with OLLAMA MODEL
# USING STREAM IS NOT THE NORMAL FUNCTION ITS NOW SPECIAL FUNCTION CALLED GENERATOR
def stream_output_ollama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]

    stream=ollama.chat.completions.create(model=MODEL_OLLAMA,messages=messages,stream=True)
    result=''
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result 


In [44]:
input_details=gr.Textbox(label='Your Message', info="Please Enter you message here", lines=2)
output_details=gr.Markdown(label='Response')

view_ui=gr.Interface(
    fn=stream_output_ollama,
    title="Ollama Model",
    inputs=[input_details],
    outputs=[output_details],
     examples=[
        "Explain the Transformer architecture in simple words",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ],
        flagging_mode='never'
)
view_ui.launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# let us do same with Gemma3:270m
GIMINI_BASE_URL= "http://localhost:11434"
MODEL_GEMMNI = "gemma3:270m"
 

In [ ]:
# create a function and get response in streams
# USING STREAM IS NOT THE NORMAL FUNCTION ITS NOW SPECIAL FUNCTION CALLED GENERATOR

def stream_output_gemma(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]

    stream=client.chat(model=MODEL_GEMMNI,messages=messages,stream=True)
    result=''
    for chunk in stream:
        result += chunk['message']['content'] or ""
        yield result 

In [46]:
input_details = gr.Textbox(label="Your message:", info="Enter a message for Claude 4.5 Sonnet", lines=7)
output_details = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_output_gemma,
    title="GIMINI", 
    inputs=[input_details], 
    outputs=[output_details], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


In [54]:
# lets create it more useful with model selection
# create function based on model selection we call function inside this main function

def model_selection(prompt,model):
    if model == 'OLLAMA':
        result= stream_output_ollama(prompt)
    elif model == "GEMMNI":
        result= stream_output_gemma(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result    



In [55]:
input_details = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["OLLAMA", "GEMMNI"], label="Select model",value="OLLAMA")
output_details = gr.Markdown(label="Response:")

view_ui = gr.Interface(
    fn=model_selection,
    title="LLMs", 
    inputs=[input_details, model_selector], 
    outputs=[output_details], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "OLLAMA"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "GEMMNI"]
        ], 
    flagging_mode="never"
    )
view_ui.launch()

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.


# Business Broucher Using Gradio UI

In [61]:
# create a function for calling function based on model selection 

def stream_model_broucher(company_name,url,model):
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model == "Ollama":
        res = stream_output_ollama(prompt)
    elif model == 'Gemmni':
        res = stream_output_gemma(prompt)
    else:
        raise ValueError("Unknown model")
    yield from res



In [62]:
name_input = gr.Textbox(label='Company Name')
url_input = gr.Textbox(label='Landing page URL including http:// or https://"')
model_selection=gr.Dropdown(['Ollama', 'Gemmni'], label="select model")
output = gr.Markdown(label='Response')

view_ui = gr.Interface(
    fn=stream_model_broucher,
    title="Business Broucher",
    inputs=[name_input,url_input,model_selection],
    outputs=[output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "Ollama"],
            ["Edward Donner", "https://edwarddonner.com", "Gemmni"]
        ],
    flagging_mode='never'
)
view_ui.launch()

* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.
